# Logistic Regression, end to end — insurance claim data (10-step ML pipeline)

This notebook follows the **10-step ML pipeline** from the course diagram:

`Business Understanding → Data Collection → Data Understanding → Data Preparation → Model Building → Model Training → Model Testing → Model Evaluation → Model Deployment → CI/CD`

Same underlying work as `02-Linear Regression/scripting.ipynb` — different algorithm, different data, identical 10-step shape. Refer back to `03-Logistic Regression/theory.md` for the math (Sigmoid, Log Loss, Gradient Descent) behind what `sklearn` does automatically in Step 6 below.

**Dataset:** `assets/claim.csv` — an insurance bodily-injury claims dataset, 1340 rows.

---
## 1. Business Understanding

**The business question:** *given what we know about a claim (claimant's age, sex, whether they had insurance, whether they wore a seatbelt, and the claim's loss amount), can we predict whether the claimant will retain an attorney?*

Why this matters in the real world: claims where the claimant hires an attorney tend to be costlier and more contested for the insurer. If an insurer can predict *early* which claims are likely to involve an attorney, they can route those claims to more experienced adjusters, set aside more reserve money, or handle them differently from the start.

**Why Logistic Regression specifically:** `ATTORNEY` (1 = retained an attorney, 0 = did not) is a **category**, not a number — recall Chapter 1: categorical target → Classification, not Regression. It's a binary category (only 2 possible values), which is exactly what Chapter 3's Logistic Regression handles. This directly follows Chapter 1's recap checklist for approaching a business problem.

---
## 2. Data Collection

Already provided as `claim.csv`. In a real project, this would come from an insurer's claims-management system, and would need its own checks (how was `ATTORNEY` actually recorded — at what point in the claim's life? is the data representative of current claims, or years out of date?).

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../assets/claim.csv")
df.head()


---
## 3. Data Understanding

Before touching a model: what do we actually have? Shape, types, missing values, and how the target and features look.

In [ ]:
print(df.shape)   # (1340, 7)
df.info()


**Reading `df.info()`:**

| Column | What it is |
|---|---|
| `CASENUM` | A case/claim ID — not a real feature (recall Chapter 1's nominal-data trap: an ID is nominal, arithmetic on it means nothing) |
| `ATTORNEY` | 1 = claimant retained an attorney, 0 = did not — **this is our target** |
| `CLMSEX` | Claimant's sex (1/0) |
| `CLMINSUR` | Whether the claimant's own car was insured (1/0) |
| `SEATBELT` | Whether the claimant was wearing a seatbelt (1/0) |
| `CLMAGE` | Claimant's age |
| `LOSS` | The claim's total economic loss, in $1000s |

`CLMSEX`, `CLMINSUR`, `SEATBELT`, `CLMAGE` show up as `float64` instead of `int64`, even though they're flag/whole-number columns — that's pandas' tell that these columns contain missing values (a column with even one `NaN` can't stay an integer dtype). That's the first clue something needs handling in Step 4, before even running `.isnull()`.

In [ ]:
df.isnull().sum()
# CASENUM       0
# ATTORNEY      0
# CLMSEX       12
# CLMINSUR     41
# SEATBELT     48
# CLMAGE      189
# LOSS          0


In [ ]:
df["ATTORNEY"].value_counts()
# 0    685
# 1    655
# -> fairly balanced (not 99/1 like fraud detection) -- good, a lopsided target
#    would need extra handling (e.g. class weighting) we don't need to worry about here


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df, x="ATTORNEY", y="CLMAGE", ax=axes[0])
axes[0].set_title("Age vs. Attorney")

sns.boxplot(data=df, x="ATTORNEY", y="LOSS", ax=axes[1])
axes[1].set_ylim(0, 20)   # zoom in -- LOSS has extreme outliers that would flatten the plot otherwise
axes[1].set_title("Loss amount vs. Attorney (zoomed in)")

plt.tight_layout()
plt.show()


**Reading these plots:** if a feature's distribution looks *identical* regardless of `ATTORNEY`'s value, that feature probably carries little predictive signal on its own. If the boxes are shifted noticeably, that's a hint the feature matters — full interpretation comes in Step 8, after training.

---
## 4. Data Preparation

Recall Step 3: 4 columns have missing values, and `CASENUM` isn't a real feature. Every missing value needs a deliberate decision, not a silent default.

**The decision:** drop rows with missing values, or fill them in (imputation)? `CLMAGE` alone is missing ~14% of rows; combined across all 4 columns, up to 244 rows have at least one gap. We'll **drop rows with any missing value** here — simple, transparent, and we still keep ~82% of the data (1096 of 1340 rows). A more advanced pass could impute `CLMAGE` with the median (recall `statistics-self-learning`: median over mean for skewed data) instead of dropping those rows — worth trying later.

In [ ]:
df_clean = df.drop(columns=["CASENUM"])   # CASENUM is an ID, not a feature -- drop it now
df_clean = df_clean.dropna()               # drop any row missing ANY remaining value

print(f"Rows before: {len(df)}, rows after: {len(df_clean)}")
# Rows before: 1340, rows after: 1096


In [ ]:
from sklearn.model_selection import train_test_split

X = df_clean.drop(columns=["ATTORNEY"])   # features: CLMSEX, CLMINSUR, SEATBELT, CLMAGE, LOSS
y = df_clean["ATTORNEY"]                   # target: 0 or 1

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,   # keep the ~51/49 ATTORNEY ratio consistent across both train and test
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
# Train: (876, 5), Test: (220, 5)


**Why `stratify=y` here but not in the Linear Regression notebook?** `stratify` preserves a *class ratio* — only meaningful for a categorical target like `ATTORNEY`. MPG (Chapter 2's target) is a continuous number with no "classes" to balance, so `stratify` doesn't apply there.

---
## 5. Model Building

Choosing and instantiating the algorithm, before any data touches it. Since Step 1 established this is Supervised + Classification + binary, **Logistic Regression** is the choice — same reasoning shape as Chapter 1's recap checklist.

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)   # max_iter: how many Gradient Descent steps to allow before giving up
# Untrained at this point -- no m's, no b yet. That's Step 6.


---
## 6. Model Training

This is where `03-Logistic Regression/theory.md` becomes real code. `sklearn`'s `LogisticRegression` computes `z = m1*x1 + m2*x2 + ... + b`, squashes it through Sigmoid into a probability, and uses **Gradient Descent** (not a direct formula like OLS — theory.md Section 4 explains why) to find the `m`'s and `b` that minimize Log Loss across the training data.

In [ ]:
model.fit(X_train, y_train)

for feature, m in zip(X.columns, model.coef_[0]):
    print(f"{feature:>10}: m = {m:+.4f}")
print(f"{'intercept':>10}: b = {model.intercept_[0]:+.4f}")
# CLMSEX: m = +0.4193
# CLMINSUR: m = +0.7992
# SEATBELT: m = -0.4963
# CLMAGE: m = +0.0055
# LOSS: m = -0.3881
# intercept: b = -0.3281


---
## 7. Model Testing

Run the trained model on the held-out test set and generate predictions — both the raw probability (theory.md Section 2, Step 2) and the final thresholded category (Step 3). Judging whether these are *good* predictions comes next, in Step 8.

In [ ]:
y_pred = model.predict(X_test)               # final 0/1 prediction (threshold at 0.5)
y_proba = model.predict_proba(X_test)[:, 1]    # raw probability before thresholding

comparison = pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred,
    "probability_of_attorney": y_proba.round(3),
})
comparison.head(10)


---
## 8. Model Evaluation

A number like "67% accuracy" means nothing alone — always compare against a baseline, and use multiple metrics, since accuracy alone hides how mistakes are distributed.

In [ ]:
# Baseline: what if we just always predicted the MAJORITY class, ignoring every feature?
majority_class = y_train.mode()[0]
baseline_accuracy = (y_test == majority_class).mean()

print(f"Majority class in training data: {majority_class}")
print(f"Baseline accuracy (always predict {majority_class}): {baseline_accuracy:.3f}")
# Majority class in training data: 0
# Baseline accuracy (always predict 0): 0.527


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

accuracy = accuracy_score(y_test, y_pred)
print(f"Model accuracy:   {accuracy:.3f}")
print(f"Baseline accuracy: {baseline_accuracy:.3f}")
print(f"Improvement:       {accuracy - baseline_accuracy:+.3f}")
# Model accuracy:   0.673
# Baseline accuracy: 0.527
# Improvement:       +0.145


In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Predicted: No Attorney", "Predicted: Attorney"],
            yticklabels=["Actual: No Attorney", "Actual: Attorney"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


**Reading a confusion matrix:** top-left = correctly predicted "No"; bottom-right = correctly predicted "Yes"; top-right = false alarm (False Positive); bottom-left = missed case (False Negative). Which mistake is worse depends on the business (Chapter 1's recap, Step 5) — missing a case that *will* involve an attorney (a False Negative) probably costs the insurer more than over-preparing for a simple claim (a False Positive).

In [ ]:
print(classification_report(y_test, y_pred, target_names=["No Attorney", "Attorney"]))


In [ ]:
auc = roc_auc_score(y_test, y_proba)
print(f"ROC-AUC: {auc:.3f}")
# ROC-AUC: 0.745
# 0.5 = no better than random guessing, 1.0 = perfect separation.
# 0.745 = a genuinely useful, if imperfect, ability to rank "more likely to hire an
# attorney" claims above "less likely" ones.


### Interpreting the coefficients

Recall Step 6's learned `m`'s (each says how `z` moves before Sigmoid squashes it into a probability — a **positive** `m` pushes the probability of "Attorney" up, **negative** pushes it down):

- **`CLMINSUR` (+0.80, strongest effect):** claimants who *had* insurance were **more** likely to retain an attorney.
- **`SEATBELT` (-0.50):** wearing a seatbelt is associated with a **lower** likelihood of attorney involvement — plausibly less severe injuries, less to dispute.
- **`CLMSEX` (+0.42):** one sex codes as more likely to retain an attorney, holding other factors constant.
- **`LOSS` (-0.39):** counter-intuitively, *higher* loss is associated with **lower** predicted attorney probability, holding other features constant — a raw statistical association in this dataset, not a causal claim.
- **`CLMAGE` (+0.0055, tiny):** a small per-year effect, though `CLMAGE` also has a much wider range (0-95) than the 0/1 features, so its total effect isn't automatically negligible.

**Important caveat:** these are associations the model found, not proven causes. Deciding *why* a pattern exists, and whether it's safe to act on, is a human/business judgment call, not something the model itself proves — same caution flagged for the unsupervised clustering example back in Chapter 1.

---
## 9. Model Deployment

Save the trained model as a standalone file, so a separate application (a UI, an API) can load and use it without this notebook, the training code, or the training data ever again.

In [ ]:
import joblib

joblib.dump(model, "logistic_regression_attorney_model.pkl")
print("Model saved to logistic_regression_attorney_model.pkl")


In [ ]:
# Simulate a separate application loading the file fresh and predicting on a new claim.
loaded_model = joblib.load("logistic_regression_attorney_model.pkl")

new_claim = pd.DataFrame({
    "CLMSEX": [1], "CLMINSUR": [1], "SEATBELT": [0], "CLMAGE": [34], "LOSS": [2.5],
})
prediction = loaded_model.predict(new_claim)[0]
probability = loaded_model.predict_proba(new_claim)[0, 1]
print(f"Predicted: {'Attorney' if prediction == 1 else 'No Attorney'}  (probability: {probability:.3f})")


**What happens next in a real project (not built here):** this `.pkl` file gets wrapped in an API (e.g. a `/predict` endpoint) that a claims-management UI calls whenever a new claim comes in. That serving/scaling layer is typically where a **Data Scientist's job ends and an ML/Software Engineer's job continues** — the DS is responsible for Steps 1-8 (a correct, well-evaluated model), not usually the serving infrastructure itself.

---
## 10. CI/CD (Continuous Integration / Continuous Deployment)

Not something run inside a notebook — this is the engineering practice wrapping the whole pipeline once it's a real, ongoing system rather than a one-off run.

- **CI:** whenever training code or data changes, automatically re-run tests — does the pipeline still run end-to-end? does the retrained model still clear a minimum quality bar (e.g. accuracy above some threshold, or ROC-AUC above 0.7)? — before anything is accepted.
- **CD:** once a new model passes those checks, automatically package (Step 9's `.pkl`) and push it live, replacing the old one, without a human manually repeating Steps 1-9 each time.

**Why it matters — model drift:** claim patterns, insurance products, and even legal/regulatory environments shift over time. A model trained once on old claims data will slowly get less accurate on new claims — this is **model drift**. CI/CD makes retraining and reshipping cheap and routine (e.g. on a schedule, as fresh claims data comes in) instead of a manual, easy-to-forget process.

---
## Recap — the 10 steps, what we actually did at each one

1. **Business Understanding** — predict `ATTORNEY` (binary) from claim features; a classification problem.
2. **Data Collection** — `claim.csv`, already provided.
3. **Data Understanding** — found missing values via dtypes before even checking `.isnull()`, confirmed the target is reasonably balanced.
4. **Data Preparation** — dropped the ID column and rows with missing values (1340→1096), stratified train/test split.
5. **Model Building** — chose and instantiated `LogisticRegression` (untrained).
6. **Model Training** — `.fit()` — Sigmoid + Log Loss + Gradient Descent, per `theory.md`.
7. **Model Testing** — generated predictions and probabilities on the held-out test set.
8. **Model Evaluation** — accuracy vs. a majority-class baseline, confusion matrix, precision/recall, ROC-AUC, and plain-English coefficient interpretation with an association-vs-causation caveat.
9. **Model Deployment** — saved the trained model as a `.pkl` file, proved a fresh process can load and use it, flagged the DS→engineering handoff point.
10. **CI/CD** — explained conceptually: the automation that makes retraining/reshipping safe and routine as claims data drifts, rather than a manual one-off process.